# Automatic NCERT query routing on Kaggle GPU
Attach the private preparation and embedding outputs. Enter a learner question below. The notebook ranks source-linked chapter and section candidates; candidate content is not approved for teaching. Check the evaluation examples before using a selected route.


In [1]:
%pip -q install 'transformers==5.22.2' 'huggingface_hub' 'accelerate'

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib, json, re
from pathlib import Path
import numpy as np
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

assert torch.cuda.is_available(), 'Enable Kaggle GPU'
root = Path('/kaggle/input')
sources = list(root.rglob('ncert_prepared/candidate_passages.jsonl'))
vectors = list(root.rglob('ncert_embeddings/candidate_vectors.npz'))
manifests = list(root.rglob('ncert_embeddings/embedding_manifest.json'))
assert len(sources) == len(vectors) == len(manifests) == 1, 'Attach one matching prepared and embedded release'
source_bytes = sources[0].read_bytes()
manifest = json.loads(manifests[0].read_text())
assert hashlib.sha256(source_bytes).hexdigest() == manifest['source_sha256']
assert hashlib.sha256(vectors[0].read_bytes()).hexdigest() == manifest['vectors_sha256']
passages = [json.loads(line) for line in source_bytes.decode('utf-8').splitlines()]
with np.load(vectors[0]) as saved:
    matrix = saved['vectors']
    ids = saved['passage_ids'].tolist()
assert matrix.shape == (len(passages), 1024) and ids == [p['passage_id'] for p in passages]
assert np.isfinite(matrix).all() and np.allclose(np.linalg.norm(matrix, axis=1), 1, atol=1e-5)
assert all(p['review_status'] == 'candidate' for p in passages)
def searchable_passage(p):
    text = re.sub(r'!\[[^]]*\]\([^)]*\)', '', p['text']).strip()
    answer_heading = re.match(r'(?i)^(?:ans(?:wer)?s?|solutions?)(?:[.:\s]|$)', p['heading'].strip())
    return 'answer_risk' not in p['review_flags'] and not answer_heading and len(text) >= 80

assert not searchable_passage({'text': '# Ans: 4', 'heading': 'Ans:', 'review_flags': []})
assert searchable_passage({'text': 'Fractions can be compared by finding a common denominator. ' * 2, 'heading': 'Fractions', 'review_flags': []})
searchable = np.array([searchable_passage(p) for p in passages])
assert searchable.any(), 'No safe routing candidates'
print('Searchable passages:', int(searchable.sum()), 'of', len(passages))
assert manifest['model_id'] == 'Qwen/Qwen3-Embedding-0.6B'
model_dir = snapshot_download(manifest['model_id'], revision=manifest['model_revision'])
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=False)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModel.from_pretrained(model_dir, dtype=torch.float32, device_map={'': 'cuda'}, attn_implementation='eager', local_files_only=True, trust_remote_code=False, use_safetensors=True).eval()
model.config.use_cache = False


Searchable passages: 6207 of 7221


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [3]:
def route(question, limit=5, min_score=0.6):
    assert isinstance(question, str) and question.strip(), 'Enter a question'
    encoded = tokenizer([manifest['query_prefix'] + question.strip()], padding=True, truncation=False, return_tensors='pt')
    assert encoded['input_ids'].shape[1] <= 32768, 'Question exceeds encoder context'
    with torch.inference_mode():
        encoded = encoded.to('cuda')
        hidden = model(**encoded).last_hidden_state
        last = encoded['attention_mask'].shape[1] - 1
        query = torch.nn.functional.normalize(hidden[:, last].float(), dim=1).cpu().numpy()[0]
    scores = matrix @ query
    scores[~searchable] = -np.inf
    ranked = np.argsort(-scores)
    # shortcut: this threshold is based on a small NCERT question set, recalibrate before learner use.
    if scores[ranked[0]] < min_score: return []
    chapters = {}
    for index in ranked:
        if not np.isfinite(scores[index]): break
        passage = passages[int(index)]
        key = (passage['grade'], passage['subject'], passage['book_code'], passage['chapter'])
        if key not in chapters:
            chapters[key] = {'grade': key[0], 'subject': key[1], 'book_code': key[2],
                             'chapter': key[3], 'score': round(float(scores[index]), 4),
                             'heading': passage['heading'], 'page': passage['page'],
                             'source_url': passage['source_url'], 'passage_id': passage['passage_id']}
        if len(chapters) >= limit: break
    return list(chapters.values())

# These two chapters are present in the pilot; rerun with a wider test set for the full corpus.
checks = [('How do I add integers below zero?', 6, 'maths', 10),
          ('How do I compare tenths and hundredths?', 7, 'maths', 3)]
for question, grade, subject, chapter in checks:
    top = route(question)[0]
    print(question, '->', top['grade'], top['subject'], top['chapter'], top['heading'])
    assert (top['grade'], top['subject'], top['chapter']) == (grade, subject, chapter)
off_topic = ('Who won today’s cricket match?', 'Write a poem about my birthday.',
             'What is the weather forecast tomorrow?', 'Book a flight to Mumbai for Friday.',
             'What is the current price of a bank stock?', 'How do I repair my laptop battery?',
             'Who is the current president of France?', 'Give me a recipe for pasta.',
             'Why does my Python program crash?', 'Which movie should I watch tonight?')
for question in off_topic:
    raw_top = route(question, min_score=0)[0]
    print('Off-topic probe:', question, 'score', raw_top['score'], 'rejected', not route(question))

benchmark = [
    ('How can I find the perimeter of a rectangle?', 6, 'maths', 6),
    ('Which objects are attracted to magnets?', 6, 'science', 4),
    ('How does a torch light up in an electric circuit?', 7, 'science', 3),
    ('What is a square number?', 8, 'maths', 1),
    ('How does air pressure cause winds and storms?', 8, 'science', 6),
    ('How does sound travel as a wave?', 9, 'science', 10),
    ('How do I graph a linear equation in two variables?', 9, 'maths', 13),
    ('How do I calculate the probability of an event?', 10, 'maths', 14),
    ('How do acids react with bases to form salts?', 10, 'science', 2),
    ('How do I calculate electric current in a circuit?', 10, 'science', 11),
    ('How can I compare two equal shares as fractions?', 6, 'maths', 7),
    ('Why does water change from liquid to vapour?', 6, 'science', 8),
    ('How do I multiply two fractions?', 7, 'maths', 8),
    ('How is heat transferred through materials?', 7, 'science', 7),
    ('How do I express a fraction as a percentage?', 8, 'maths', 8),
    ('How do mirrors and lenses form images?', 8, 'science', 10),
    ('How do I find the surface area of a cuboid?', 9, 'maths', 14),
    ('What are cells and their organelles?', 9, 'science', 2),
    ('What is the sine of an angle in a right triangle?', 10, 'maths', 8),
    ('How does an electric current create a magnetic field?', 10, 'science', 12),
]
results = []
for question, grade, subject, chapter in benchmark:
    matches = route(question)
    rank = next((i + 1 for i, row in enumerate(matches) if (row['grade'], row['subject'], row['chapter']) == (grade, subject, chapter)), None)
    results.append(rank)
    print('Benchmark:', question, 'expected', (grade, subject, chapter), 'rank', rank, 'top score', matches[0]['score'] if matches else None)
print('Benchmark top-1:', sum(rank == 1 for rank in results), '/', len(results),
      'top-5:', sum(rank is not None for rank in results), '/', len(results))
assert all(not route(question) for question in off_topic), 'Off-topic rejection needs recalibration'
assert sum(rank is not None for rank in results) >= 15, 'Topic routing needs review'


How do I add integers below zero? -> 6 maths 10 Brahmagupta’s Rules for Subtraction (Brāhma-sphuṭa-siddhānta 18.31-18.32)
How do I compare tenths and hundredths? -> 7 maths 3 Chapter – 3 A Peek Beyond the Point
Off-topic probe: Who won today’s cricket match? score 0.5369 rejected True
Off-topic probe: Write a poem about my birthday. score 0.4251 rejected True
Off-topic probe: What is the weather forecast tomorrow? score 0.4798 rejected True
Off-topic probe: Book a flight to Mumbai for Friday. score 0.4566 rejected True
Off-topic probe: What is the current price of a bank stock? score 0.4908 rejected True
Off-topic probe: How do I repair my laptop battery? score 0.4929 rejected True
Off-topic probe: Who is the current president of France? score 0.3032 rejected True
Off-topic probe: Give me a recipe for pasta. score 0.4271 rejected True
Off-topic probe: Why does my Python program crash? score 0.3959 rejected True
Off-topic probe: Which movie should I watch tonight? score 0.4074 rejected 

In [4]:
question = 'How do I compare tenths and hundredths?'
print(json.dumps(route(question), indent=2))


[
  {
    "grade": 7,
    "subject": "maths",
    "book_code": "gegp1",
    "chapter": 3,
    "score": 0.7603,
    "heading": "Chapter \u2013 3 A Peek Beyond the Point",
    "page": 44,
    "source_url": "https://ncert.nic.in/textbook/pdf/gegp103.pdf",
    "passage_id": "2f7f893911cc34ff89d0d020"
  },
  {
    "grade": 7,
    "subject": "maths",
    "book_code": "gegp2",
    "chapter": 12,
    "score": 0.6595,
    "heading": "4.2 Decimal Multiplication",
    "page": 7,
    "source_url": "https://ncert.nic.in/textbook/pdf/gegp204.pdf",
    "passage_id": "f6746c194da1dcdd6e49ba9c"
  },
  {
    "grade": 6,
    "subject": "maths",
    "book_code": "fegp1",
    "chapter": 7,
    "score": 0.6164,
    "heading": "Group 1",
    "page": 21,
    "source_url": "https://ncert.nic.in/textbook/pdf/fegp107.pdf",
    "passage_id": "87118fc7e95ac0bb2dace657"
  },
  {
    "grade": 8,
    "subject": "maths",
    "book_code": "hegp2",
    "chapter": 8,
    "score": 0.5922,
    "heading": "Expressing Fracti